# Evaluation

Measured **feasibility rate** and **optimality gap** on the held-out test set, for any trained adapter (SFT or reward), with single-shot and best-of-N decoding.

- **Feasibility:** fraction of outputs that are valid tours (a permutation of all cities).
- **Optimality gap:** over feasible outputs, mean % by which the tour exceeds the OR-Tools optimum.

A **size-split** (10–13 vs 14–20 cities) is reported because it was the key diagnostic — it revealed the 0.5B model failed completely on large instances.

**Before running:**
1. T4 GPU runtime.
2. Upload `test.jsonl` and `serialize.py` into `/content/`.
3. Set `ADAPTER` below to the model you want to evaluate.

## 1. Installing + reloading the model (16-bit, merged for speed)

In [ ]:
!pip install -q -U "transformers>=4.46" peft accelerate bitsandbytes "protobuf>=5.26,<6"
!pip uninstall -q -y torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER = "/content/drive/MyDrive/llm-tsp/reward-adapter-1.5b"   # or sft-adapter-1.5b

tokenizer = AutoTokenizer.from_pretrained(ADAPTER)
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16,
                                            device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER)
model = model.merge_and_unload()
print("model ready")

## 2. Helpers

In [ ]:
import json, numpy as np
from serialize import parse_tour, is_feasible, SYSTEM_PROMPT, make_prompt

def tour_length(coords, tour):
    pts = coords[tour]; nxt = np.roll(pts, -1, axis=0)
    return float(np.sqrt(((pts - nxt) ** 2).sum(axis=1)).sum())

def build_messages(coords):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": make_prompt(np.array(coords))}]

def generate_for(coords):
    msgs = build_messages(coords)
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**inp, max_new_tokens=256, do_sample=True,
                         temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)

test = [json.loads(l) for l in open('/content/test.jsonl')]
print(len(test), "test instances")

## 3. Single-shot evaluation, split by instance size

`max_new_tokens=256` so even 20-city tours are not truncated (a 128-token limit
silently truncated long tours in an earlier run and deflated feasibility).

In [ ]:
small, large = [], []
for ex in test:
    coords = np.array(ex['coords']); n = len(coords)
    tour = parse_tour(generate_for(coords), n)
    ok = is_feasible(tour, n)
    gap = (tour_length(coords, tour) - ex['opt_len'])/ex['opt_len']*100 if ok else None
    (small if n <= 13 else large).append((ok, gap))

for label, group in [("10-13 cities", small), ("14-20 cities", large)]:
    fr = sum(ok for ok,_ in group)/len(group)*100
    gaps = [g for ok,g in group if ok]
    mg = np.mean(gaps) if gaps else float('nan')
    print(f"{label}: feasibility {fr:.1f}%  |  gap {mg:.2f}%  ({len(group)} instances)")

## 4. Best-of-N evaluation (inference-time scaling)

Generated N tours per instance and kept the shortest feasible one. This exploits the verifiable reward at inference time and is the most effective cheap lever for lowering the optimality gap. Slow (Nx generations) — run on a subset for a quick read.

In [ ]:
def best_of_n(coords, n_samples=8):
    n = len(coords); best = float('inf')
    for _ in range(n_samples):
        tour = parse_tour(generate_for(coords), n)
        if is_feasible(tour, n):
            best = min(best, tour_length(np.array(coords), tour))
    return best if best < float('inf') else None

subset = test[:60]      # subset to keep runtime reasonable
feas, gaps = 0, []
for i, ex in enumerate(subset):
    L = best_of_n(np.array(ex['coords']), 8)
    if L is not None:
        feas += 1
        gaps.append((L - ex['opt_len'])/ex['opt_len']*100)
    if (i+1) % 20 == 0:
        print(f"  ...{i+1}/{len(subset)}")

print(f"\nBest-of-8 (n={len(subset)}) | Feasibility: {feas/len(subset)*100:.1f}%  |  Gap: {np.mean(gaps):.2f}%")